In [1]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from meter.modules.heads import Pooler

from torch.utils.data import DataLoader
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule

from torch.optim import AdamW

from transformers import AutoConfig
from transformers import ElectraTokenizer, ViltFeatureExtractor
from transformers import AutoProcessor, AutoImageProcessor, AutoTokenizer
from transformers import AutoModel, AutoModelForSequenceClassification

# from refcoco_utils import get_bounded_subimage
# from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

In [2]:
def _loss_names(d):
    ret = {
        "itm": 0,
        "mlm": 0,
        "mpp": 0,
        "vqa": 0,
        "vcr": 0,
        "vcr_qar": 0,
        "nlvr2": 0,
        "irtr": 0,
        "contras": 0,
        "snli": 0,
        "ref": 0,
        "mrpc" : 1,
        "rte" : 0,
        'wnli' : 0,
        'sst2' : 0,
        'qqp' : 0,
        'qnli' : 0,
        'mnli' : 0,
        'cola' : 0,
        'cifar10' : 0
    }
    ret.update(d)
    return ret

config = {  
    "exp_name":"finetune_mrpc",
    "seed" : 42,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    # "datasets" : ["coco", "vg"],
    "datasets" : ["snli"],
    # 'loss_names' : _loss_names({"itm": 1, "mlm": 1}),
    'loss_names' : _loss_names({"snli": 1}),
    "batch_size" : 1,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.
    "model_type" : 'two-tower',
    
    # One-Tower Settings
    "random_init_encoder" : False,
    "encoder" : "facebook/deit-tiny-patch16-224",
    'encoder_type' : 'image',
    # Transformer Setting
    # 'vit' : "vit_base_patch32_384",
    'hidden_size' : 192,
    'num_heads' : 12,
    'num_layers' : 12,
    'mlp_ratio' : 4,
    'drop_rate' : 0.1,
    

    # Image setting
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    "random_init_vision_encoder" : False,
    "image_encoder_hidden_size" : 192,
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],

    # Text Setting
    "text_encoder" : "google/electra-small-discriminator",
    "random_init_text_encoder" : False,
    "text_encoder_hidden_size" : 256,
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,
    "vqav2_label_size" : 3129,
    "max_text_len" : 128,

    'pooler_type' : 'double', 

    # CrossLayer Setting
    "num_cross_layers" : 6,
    "cross_layer_hidden_size" : 256,
    "num_cross_layer_heads" : 4,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,
    
    # Architecture Setting
    "two_tower" : False,
    "multi_modal_encoder" : 'dandelin/vilt-b32-mlm',
    
    
    
    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 5e-5,
    "weight_decay" : 0.0,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 0,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Encoder Settings
    "freeze_image_encoder" : True,
    "freeze_text_encoder" : False,
    'freeze_cross_modal_layers' : True,
    
    'text_only' : False,
    

    # Downstream Setting
    "get_recall_metric" : False,
    
    'freeze' : True,
    
    # "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : "/home/claytonfields/nlp/code/meter/data/arrow",
    "log_dir" : "result",
    "per_gpu_batchsize" : 2,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : 1,
    "num_nodes" : 1,
    'load_path' : '',
    # "load_path" : "/home/claytonfields/nlp/code/meter/result/mlm_itm_seed0_from_/meter_electra_small_deit_tiny_p16_is224_bs288_is1M/checkpoints/epoch=43-step=898039.ckpt",
    # "load_path" : '/home/claytonfields/nlp/code/meter/result/mlm_itm_deit_fr_electra_fr_is224_ps16_bs336_pgbs84_ts100k/checkpoints/epoch=5-step=96215.ckpt',
    "num_workers" : 12,
    "precision" : 32
}


In [3]:
dm = MTDataModule(config, dist=False)

In [5]:
model = METERTransformerSS(config)
model

Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-tiny-patch16-224 and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


METERTransformerSS(
  (cross_modal_text_transform): Linear(in_features=256, out_features=256, bias=True)
  (cross_modal_image_transform): Linear(in_features=192, out_features=256, bias=True)
  (cross_modal_image_layers): ModuleList(
    (0-5): 6 x BertCrossLayer(
      (attention): BertAttention(
        (self): BertSelfAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (output): BertSelfOutput(
          (dense): Linear(in_features=256, out_features=256, bias=True)
          (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (crossattention): BertAttention(
        (self): BertSelfAttention(
          (query): Linear(in_features=256, out_features=256, bias=

In [6]:
dm.prepare_data()
dm.setup('train')

In [7]:
dl = dm.train_dataloader()

In [8]:
batch = next(iter(dl))

You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenize

In [9]:
batch

{'labels': [0, 2],
 'text': ['Two old men robbing a convenience store.',
  'Two humans in a store.'],
 'image': [tensor([[[[ 1.1529,  0.7248,  0.6906,  ...,  0.5193,  0.2111,  0.7248],
            [ 0.4166, -0.0116,  1.1872,  ...,  0.4508,  0.2624,  0.6392],
            [-0.5424, -0.7479, -0.3369,  ...,  0.5536,  0.2624,  0.7591],
            ...,
            [ 0.5022,  0.5364,  0.5364,  ..., -1.1932, -1.1418, -1.1589],
            [ 0.7762,  0.7591,  0.7419,  ..., -0.8849, -0.8678, -0.9192],
            [ 0.9474,  0.8789,  0.8618,  ..., -0.0801,  0.0569,  0.0056]],
  
           [[ 0.9930,  0.6604,  0.7654,  ...,  0.8354,  0.4678,  0.9230],
            [ 0.5028,  0.1527,  1.2731,  ...,  0.7304,  0.5203,  0.8880],
            [-0.5651, -0.5301, -0.1625,  ...,  0.8354,  0.5028,  1.0280],
            ...,
            [ 0.5903,  0.6254,  0.6254,  ..., -1.1429, -1.1078, -1.0903],
            [ 0.8704,  0.8529,  0.8354,  ..., -0.7052, -0.7052, -0.7752],
            [ 1.0455,  0.9755,  0.958

In [11]:
from transformers.models.vilt.modeling_vilt import  ViltEmbeddings
from transformers.models.vilt.configuration_vilt import ViltConfig

In [13]:
vilt_config = ViltConfig(config)

In [14]:
ViltEmbeddings(vilt_config)

TypeError: empty() received an invalid combination of arguments - got (tuple, dtype=NoneType, device=NoneType), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.memory_format memory_format, torch.dtype dtype, torch.layout layout, torch.device device, bool pin_memory, bool requires_grad)
 * (tuple of ints size, *, torch.memory_format memory_format, Tensor out, torch.dtype dtype, torch.layout layout, torch.device device, bool pin_memory, bool requires_grad)


ViltConfig {
  "attention_probs_dropout_prob": 0.0,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_image_length": -1,
  "max_position_embeddings": 40,
  "modality_type_vocab_size": 2,
  "model_type": "vilt",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "num_images": -1,
  "patch_size": 32,
  "qkv_bias": true,
  "tie_word_embeddings": false,
  "transformers_version": "4.36.2",
  "type_vocab_size": 2,
  "vocab_size": {
    "batch_size": 1,
    "cross_layer_drop_rate": 0.1,
    "cross_layer_hidden_size": 256,
    "cross_layer_mlp_ratio": 4,
    "data_root": "/home/claytonfields/nlp/code/meter/data/arrow",
    "datasets": [
      "snli"
    ],
    "decay_power": 1,
    "draw_false_image": 1,
    "draw_false_text": 0,
    "drop_rate": 0.1,
    "encoder": "facebook/deit-tiny-patch16-224",
    "encoder_type": "image",
 